In [107]:
import json
import re
from pathlib import Path
from collections import defaultdict
import numpy as np

# -------------------------
# Config
# -------------------------
PROJECT_ROOT = Path("~/mack/yyin5/HUGSIM").expanduser()
BENCH_ROOT = PROJECT_ROOT / "outputs" / "benchmark_new_2"
MODEL_NAME = "dynamo_v6_nav3" # "dynamo_v6_fixed_input" # "dynamo_v6_new_scorer_v2_cautious_regularized_bis"
# "private_dynamo_v6_new_scorer_v2_aggressive_regularized_ckpt20251103"  # <- change if needed
DATASETS = {"kitti360", "nuscenes", "pandaset", "waymo"}
DIFF_RE = re.compile(r"(easy|medium|hard|extreme)", re.IGNORECASE)

# canonical score order and key mapping
SCORE_KEYS = [
    ("nc", "NC"),
    ("dac", "DAC"),
    ("ttc", "TTC"),
    ("c", "COM"),
    ("rc", "RC"),
    # ("pdms", "PDMS"),
    ("hdscore", "HDSCORE"),
]

# -------------------------
# Helpers
# -------------------------
def avg_dict(acc):
    return {k: float(np.mean(v)) for k, v in acc.items() if len(v) > 0}

def ensure_floatable(d):
    out = {}
    for k, v in d.items():
        if isinstance(v, (int, float)):
            out[k] = float(v)
    return out

def print_scores(d):
    for key, display in SCORE_KEYS:
        if key in d:
            print(f"{d[key]:.4f}")

# -------------------------
# Dataset roots discovery
# -------------------------
dataset_roots = {}
for ds in DATASETS:
    candidate = BENCH_ROOT / f"{ds}_{MODEL_NAME}"
    if candidate.is_dir():
        dataset_roots[ds] = candidate

if not dataset_roots:
    raise SystemExit(f"No dataset folders found for model '{MODEL_NAME}' under {BENCH_ROOT}")

print("Found dataset roots:")
for ds, p in dataset_roots.items():
    print(f"  {ds}: {p}")

# -------------------------
# Accumulators
# -------------------------
global_totals = defaultdict(list)
global_details = defaultdict(lambda: defaultdict(list))

per_dataset_totals = defaultdict(lambda: defaultdict(list))
per_dataset_details = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

per_diff_totals = defaultdict(lambda: defaultdict(list))
per_diff_details = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

per_dataset_diff_totals = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
per_dataset_diff_details = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))

missing_per_dataset = defaultdict(list)
present_per_dataset = defaultdict(list)

# -------------------------
# Aggregate data
# -------------------------
all_eval_files = []

for ds, ds_root in dataset_roots.items():
    for scene_dir in sorted(p for p in ds_root.iterdir() if p.is_dir()):
        eval_fp = scene_dir / "eval.json"
        diff_match = DIFF_RE.search(scene_dir.name)
        diff = diff_match.group(1).lower() if diff_match else "unknown"

        if not eval_fp.exists():
            missing_per_dataset[ds].append(scene_dir)
            continue

        present_per_dataset[ds].append(scene_dir)
        all_eval_files.append(eval_fp)

        data = json.loads(eval_fp.read_text())
        top = ensure_floatable(data)

        # global
        for k, v in top.items():
            global_totals[k].append(v)
        for th, subs in data.get("details", {}).items():
            for k, v in subs.items():
                global_details[th][k].append(float(v))

        # per-dataset
        for k, v in top.items():
            per_dataset_totals[ds][k].append(v)
        for th, subs in data.get("details", {}).items():
            for k, v in subs.items():
                per_dataset_details[ds][th][k].append(float(v))

        # per-difficulty
        for k, v in top.items():
            per_diff_totals[diff][k].append(v)
        for th, subs in data.get("details", {}).items():
            for k, v in subs.items():
                per_diff_details[diff][th][k].append(float(v))

        # per (dataset,difficulty)
        for k, v in top.items():
            per_dataset_diff_totals[ds][diff][k].append(v)
        for th, subs in data.get("details", {}).items():
            for k, v in subs.items():
                per_dataset_diff_details[ds][diff][th][k].append(float(v))

# -------------------------
# Compute means
# -------------------------
global_means = avg_dict(global_totals)
global_details_mean = {th: avg_dict(sub) for th, sub in global_details.items()}
per_dataset_means = {ds: avg_dict(tot) for ds, tot in per_dataset_totals.items()}
per_dataset_details_mean = {ds: {th: avg_dict(sub) for th, sub in details.items()}
                            for ds, details in per_dataset_details.items()}
per_diff_means = {diff: avg_dict(tot) for diff, tot in per_diff_totals.items()}
per_diff_details_mean = {diff: {th: avg_dict(sub) for th, sub in details.items()}
                         for diff, details in per_diff_details.items()}
per_dataset_diff_means = {ds: {diff: avg_dict(tot) for diff, tot in diff_map.items()}
                          for ds, diff_map in per_dataset_diff_totals.items()}
per_dataset_diff_details_mean = {
    ds: {diff: {th: avg_dict(sub) for th, sub in th_map.items()}
         for diff, th_map in diff_map.items()}
    for ds, diff_map in per_dataset_diff_details.items()
}

# -------------------------
# Unified printing for figure (TSV-ready)
# -------------------------
DISPLAY_NAMES = {
    "kitti360": "KITTI360",
    "nuscenes": "nuScenes",
    "pandaset": "Pandaset",
    "waymo": "Waymo",
}
DATASET_ORDER = ["kitti360", "nuscenes", "pandaset", "waymo"]
DIFF_ORDER = ["easy", "medium", "hard", "extreme"]

def _fmt(v):
    return "" if v is None or np.isnan(v) else f"{v:.4f}"

def _get(ds, diff, key):
    try:
        return float(per_dataset_diff_means[ds][diff][key])
    except Exception:
        return np.nan

def _get_ds_mean(ds, key):
    try:
        return float(per_dataset_means[ds][key])
    except Exception:
        return np.nan

def _get_diff_mean(diff, key):
    try:
        return float(per_diff_means[diff][key])
    except Exception:
        return np.nan

def _get_global(key):
    try:
        return float(global_means[key])
    except Exception:
        return np.nan


# ----------------------------------------------------------
# 1. Per (dataset, difficulty) wide table (figure-style)
# ----------------------------------------------------------
print("\n=== PER (DATASET, DIFFICULTY) MEANS ===")
header = [f"{DISPLAY_NAMES.get(ds, ds)} {d.capitalize()}" for ds in DATASET_ORDER for d in DIFF_ORDER]
print("\t".join(header))

for key, _ in SCORE_KEYS:
    row = [_fmt(_get(ds, d, key)) for ds in DATASET_ORDER for d in DIFF_ORDER]
    print("\t".join(row))

# ----------------------------------------------------------
# 2. Per-dataset means (one value per dataset, same order)
# ----------------------------------------------------------
print("\n=== PER-DATASET MEANS ===")
header = [DISPLAY_NAMES.get(ds, ds) for ds in DATASET_ORDER]
print("\t".join(header))

for key, _ in SCORE_KEYS:
    row = [_fmt(_get_ds_mean(ds, key)) for ds in DATASET_ORDER]
    print("\t".join(row))

# ----------------------------------------------------------
# 3. Per-difficulty means (across datasets)
# ----------------------------------------------------------
print("\n=== PER-DIFFICULTY MEANS ===")
header = [d.capitalize() for d in DIFF_ORDER]
print("\t".join(header))

for key, _ in SCORE_KEYS:
    row = [_fmt(_get_diff_mean(d, key)) for d in DIFF_ORDER]
    print("\t".join(row))

# ----------------------------------------------------------
# 4. Global mean (single row, no label)
# ----------------------------------------------------------
print("\n=== GLOBAL MEAN ===")
row = [_fmt(_get_global(key)) for key, _ in SCORE_KEYS]
print("\n".join(row))

# ----------------------------------------------------------
# 5. Save all numeric means into one TSV file
# ----------------------------------------------------------
tsv_path = BENCH_ROOT / f"aggregate_eval_{MODEL_NAME}_means_for_figure.tsv"
with open(tsv_path, "w", encoding="utf-8") as f:
    # section 1
    f.write("## Per (dataset, difficulty)\n")
    f.write("\t".join(header) + "\n")
    for key, _ in SCORE_KEYS:
        row = [_fmt(_get(ds, d, key)) for ds in DATASET_ORDER for d in DIFF_ORDER]
        f.write("\t".join(row) + "\n")

    # section 2
    f.write("\n## Per-dataset means\n")
    f.write("\t".join(DATASET_ORDER) + "\n")
    for key, _ in SCORE_KEYS:
        row = [_fmt(_get_ds_mean(ds, key)) for ds in DATASET_ORDER]
        f.write("\t".join(row) + "\n")

    # section 3
    f.write("\n## Per-difficulty means\n")
    f.write("\t".join(DIFF_ORDER) + "\n")
    for key, _ in SCORE_KEYS:
        row = [_fmt(_get_diff_mean(d, key)) for d in DIFF_ORDER]
        f.write("\t".join(row) + "\n")

    # section 4
    f.write("\n## Global mean\n")
    row = [_fmt(_get_global(key)) for key, _ in SCORE_KEYS]
    f.write("\n".join(row) + "\n")

print(f"\nSaved all numeric means to: {tsv_path}")


print("\n=== Folders missing eval.json (per dataset) ===")
total_missing = 0
for ds in sorted(dataset_roots.keys()):
    missing = missing_per_dataset[ds]
    total_missing += len(missing)
    print(f"\n[{ds}]  missing: {len(missing)}")
    for d in missing:
        print(f"  {d}")

print(f"\nTotal eval.json files found: {len(all_eval_files)}")
print(f"Total missing folders: {total_missing}")

# -------------------------
# Save full JSON report
# -------------------------
report = {
    "model": MODEL_NAME,
    "averages": {
        "global": global_means,
        "per_dataset": per_dataset_means,
        "per_difficulty": per_diff_means,
        "per_dataset_difficulty": per_dataset_diff_means,
    },
    "details": {
        "global": global_details_mean,
        "per_dataset": per_dataset_details_mean,
        "per_difficulty": per_diff_details_mean,
        "per_dataset_difficulty": per_dataset_diff_details_mean,
    },
    "missing_folders": {ds: [str(p) for p in missing_per_dataset[ds]] for ds in dataset_roots.keys()},
    "present_folders": {ds: [str(p) for p in present_per_dataset[ds]] for ds in dataset_roots.keys()},
}

out_path = BENCH_ROOT / f"aggregate_eval_{MODEL_NAME}.json"
out_path.write_text(json.dumps(report, indent=2))
print(f"\nSaved aggregate report to: {out_path}")


Found dataset roots:
  nuscenes: /home/yyin5/mack/yyin5/HUGSIM/outputs/benchmark_new_2/nuscenes_dynamo_v6_nav3
  pandaset: /home/yyin5/mack/yyin5/HUGSIM/outputs/benchmark_new_2/pandaset_dynamo_v6_nav3
  waymo: /home/yyin5/mack/yyin5/HUGSIM/outputs/benchmark_new_2/waymo_dynamo_v6_nav3
  kitti360: /home/yyin5/mack/yyin5/HUGSIM/outputs/benchmark_new_2/kitti360_dynamo_v6_nav3

=== PER (DATASET, DIFFICULTY) MEANS ===
KITTI360 Easy	KITTI360 Medium	KITTI360 Hard	KITTI360 Extreme	nuScenes Easy	nuScenes Medium	nuScenes Hard	nuScenes Extreme	Pandaset Easy	Pandaset Medium	Pandaset Hard	Pandaset Extreme	Waymo Easy	Waymo Medium	Waymo Hard	Waymo Extreme
0.7637	0.5349	0.2571	0.2825	0.9737	0.3888	0.5654	0.6241	0.8940	0.5911	0.2202	0.4850	0.9322	0.7224	0.4649	0.5202
0.9345	0.9108	0.8849	0.8725	1.0000	1.0000	1.0000	1.0000	1.0000	0.9971	1.0000	1.0000	0.9657	0.9666	0.9635	0.9781
0.7470	0.5089	0.2102	0.2465	0.9437	0.3144	0.4964	0.5855	0.8588	0.5016	0.1353	0.4430	0.9152	0.6516	0.4046	0.4779
0.9399	0.9497	0.